# ENTSO-E Feature Engineering

Features are built for a forecast made around 11:00 on D-1, for delivery day D. A value is used only if it is known by then. `create_features` builds every series once. Each model family is saved in two scopes: Poland, and Poland plus the foreign addition. That is four files. `select_features(df, model, scope)` keeps one of them: `model` is `"trees"` or `"nn"`, and `scope` is `"pl"` or `"pl_foreign"`.

**Availability**

* The Polish day-ahead price of day D is the target and is not a feature. Lags, the previous-day mean, minimum and maximum, and the 7-day statistics use only earlier days.
* The day-ahead `load_forecast` for day D is kept. `PL_load_fc_delta24` is that forecast minus the forecast for the same hour of D-1.
* Wind and solar forecasts are shifted by 24 hours, because the forecast for day D is not available at decision time. Residual load keeps the load forecast for day D and subtracts those lagged wind and solar forecasts.
* Actual coal generation is shifted by 48 hours, so only D-2 is visible. Coal is hard coal, lignite and coal-derived gas, summed.
* A spread is the Polish price minus the foreign price, both lagged 24 hours. Germany's wind forecast is onshore plus offshore. Names shorten `DE_LU` to `DE` and `SE_4` to `SE4`, except `spread_PL_DE_LU_lag24`, which keeps the bidding-zone code.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.file_io import (
    load_entsoe_zones,
    save_merged_data,
    inspect_merged_file,
)

from src.utils.feature_engineering.fe_entsoe import (
    COLUMNS,
    create_features,
    select_features,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
data_entsoe = load_entsoe_zones(DATA_DIR / "preprocessed" / "entsoe" / date_range)

## 1. Input Columns

Preprocessed series per bidding zone, before any shift or aggregation.

In [3]:
for country, data in data_entsoe.items():
    print(f"=== {country} ===")
    print(f"{list(data.columns)}")

=== AT ===
['load_forecast', 'water_reservoirs_hydro_storage', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Onshore', 'generation_forecast_Scheduled_Consumption', 'generation_forecast_Scheduled_Aggregated']
=== CZ ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast', 'generation_forecast_Scheduled_Consumption', 'generation_forecast_Scheduled_Aggregated']
=== DE_LU ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Offshore', 'generation_wind_solar_forecast_Wind_Onshore', 'generation_forecast']
=== FR ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Onshore', 'generation_forecast']
=== HU ===
['load_forecast', 'day_ahead_prices', 'generation_wind_solar_forecast_Solar', 'generation_wind_solar_forecast_Wind_Onshore', 'generation_forecast']
=== LT ===
['load_forecast', 'day_ahead_p

## 2. Features

`create_features` builds 28 columns: 14 for Poland and 14 in the foreign addition. Neural networks later drop the aggregates a network can rebuild from the raw series.

**Poland — prices (8).** `PL_price_lag24`, `PL_price_lag48`, `PL_price_lag168`. From day D-1: `PL_price_d1_mean`, `PL_price_d1_min`, `PL_price_d1_max`. `PL_price_mean7d_same_hour` is the mean of this hour over D-1…D-7. `PL_price_std7d` is the rolling standard deviation over 7 days, taken at the last hour of D-1, so the same value is used for every hour of day D.

**Poland — load and renewables (5).** `PL_load_fc`, `PL_load_fc_delta24`, `PL_wind_fc_lag24`, `PL_solar_fc_lag24`. `PL_resload_persist` is `PL_load_fc` minus the two lagged renewable forecasts.

**Poland — generation (1).** `PL_coal_gen_lag48`.

**Foreign addition (14).** Spreads versus DE_LU, CZ, SK, SE4 and LT. `DE_price_lag24`. Day-ahead load for DE, CZ, SK, SE4 and LT. `DE_wind_fc_lag24` (onshore plus offshore), `DE_solar_fc_lag24`. `DE_resload_persist` is the German load forecast minus those two lagged forecasts.

In [4]:
features = create_features(data_entsoe)

In [5]:
print(features.shape)
print(list(features.columns))

(33240, 28)
['PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_price_d1_mean', 'PL_price_d1_min', 'PL_price_d1_max', 'PL_price_mean7d_same_hour', 'PL_price_std7d', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24', 'PL_solar_fc_lag24', 'PL_resload_persist', 'PL_coal_gen_lag48', 'spread_PL_DE_LU_lag24', 'spread_PL_CZ_lag24', 'spread_PL_SK_lag24', 'spread_PL_SE4_lag24', 'spread_PL_LT_lag24', 'DE_price_lag24', 'DE_load_fc', 'CZ_load_fc', 'SK_load_fc', 'SE4_load_fc', 'LT_load_fc', 'DE_wind_fc_lag24', 'DE_solar_fc_lag24', 'DE_resload_persist']


## 3. The Four Subsets

`COLUMNS` lists the four combinations. `pl` is Poland. `pl_foreign` is Poland plus the foreign addition.

Trees keep 14 Polish columns, and 28 with the foreign addition.

Neural networks keep 8 Polish columns: the three price lags, `PL_load_fc`, `PL_load_fc_delta24`, wind, solar and coal. The daily price statistics, `PL_price_std7d` and `PL_resload_persist` stay in the tree set. With the foreign addition that is 20 columns: those 8, plus the five spreads, the five foreign load forecasts, and German wind and solar. `DE_price_lag24` and `DE_resload_persist` stay in the tree set.

In [6]:
for model, scopes in COLUMNS.items():
    for scope in scopes:
        subset = select_features(features, model, scope)
        print(f"=== {model}/{scope} ===")
        print(subset.shape)
        print(list(subset.columns))

=== trees/pl ===
(33240, 14)
['PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_price_d1_mean', 'PL_price_d1_min', 'PL_price_d1_max', 'PL_price_mean7d_same_hour', 'PL_price_std7d', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24', 'PL_solar_fc_lag24', 'PL_resload_persist', 'PL_coal_gen_lag48']
=== trees/pl_foreign ===
(33240, 28)
['PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_price_d1_mean', 'PL_price_d1_min', 'PL_price_d1_max', 'PL_price_mean7d_same_hour', 'PL_price_std7d', 'PL_load_fc', 'PL_load_fc_delta24', 'PL_wind_fc_lag24', 'PL_solar_fc_lag24', 'PL_resload_persist', 'PL_coal_gen_lag48', 'spread_PL_DE_LU_lag24', 'spread_PL_CZ_lag24', 'spread_PL_SK_lag24', 'spread_PL_SE4_lag24', 'spread_PL_LT_lag24', 'DE_price_lag24', 'DE_load_fc', 'CZ_load_fc', 'SK_load_fc', 'SE4_load_fc', 'LT_load_fc', 'DE_wind_fc_lag24', 'DE_solar_fc_lag24', 'DE_resload_persist']
=== nn/pl ===
(33240, 8)
['PL_price_lag24', 'PL_price_lag48', 'PL_price_lag168', 'PL_load_fc', 'PL_load_fc

## 4. Conclusions

Poland and Poland plus the foreign addition are each saved for trees and for neural networks, four files in total. Files go to `data/features/entsoe/{trees|nn}/{pl|pl_foreign}/`, and each file is verified by reading it back.

## 5. Saving Feature Data

In [7]:
for model, scopes in COLUMNS.items():
    for scope in scopes:
        path = (
            DATA_DIR / "features" / "entsoe" / model / scope / f"{date_range}.parquet"
        )
        save_merged_data(select_features(features, model, scope), path)
        inspect_merged_file(path)

  [FILE] data/features/entsoe/trees/pl/2022-12-01_2026-09-16.parquet | shape: (33240, 14) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 22:00:00+00:00 | cols: 14 | NaN: 795
  [FILE] data/features/entsoe/trees/pl_foreign/2022-12-01_2026-09-16.parquet | shape: (33240, 28) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 22:00:00+00:00 | cols: 28 | NaN: 1017
  [FILE] data/features/entsoe/nn/pl/2022-12-01_2026-09-16.parquet | shape: (33240, 8) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 22:00:00+00:00 | cols: 8 | NaN: 362
  [FILE] data/features/entsoe/nn/pl_foreign/2022-12-01_2026-09-16.parquet | shape: (33240, 20) | idx: 2022-11-30 23:00:00+00:00-2026-09-15 22:00:00+00:00 | cols: 20 | NaN: 535
